## Meta Ads

#### Useful links:
- [Get a Developer account](https://developers.facebook.com/async/registration/dialog/?src=default)
- [Identity and Location Confirmation](Facebook.com/ID)
- [Meta Ads API Documentation](https://www.facebook.com/ads/library/api/)
- [Meta Ads API - Ads Archive Reference](https://developers.facebook.com/docs/graph-api/reference/ads_archive/)


In [1]:
# requirements
%pip install pandas==2.3.2 requests==2.32.5 python-dotenv==1.1.1 pandera==0.26.1

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import glob
import pandas as pd
import pandera.pandas as pa
from pandera.errors import SchemaErrors
import requests
from dotenv import load_dotenv
from datetime import datetime
from time import sleep

In [3]:
import sys

print("Python version:")
print(sys.version)

Python version:
3.11.10 | packaged by Anaconda, Inc. | (main, Oct  3 2024, 07:22:26) [MSC v.1929 64 bit (AMD64)]


In [4]:
# environment variables
load_dotenv()
ACCESS_TOKEN = os.getenv("ACCESS_TOKEN") # Token expires in
FILEPATH = os.getenv("FILEPATH")

#### Meta Ads API Available Parameters


- ad_active_status: *enum {ACTIVE, ALL, INACTIVE}* <span style="color:red">**(DEFAULT: Active)**</span>
- ad_delivery_date_max: *str*
- ad_delivery_date_min: *str*
- ad_reached_countries: *array<enum> (countries abbreviations)*
- ad_type: *enum {ALL, EMPLOYMENT_ADS, FINANCIAL_PRODUCTS_AND_SERVICES_ADS, HOUSING_ADS, POLITICAL_AND_ISSUE_ADS}* <span style="color:red">**(DEFAULT: ALL)**</span>
- bylines: *array<str>*
- delivery_by_region: *array<str>*
- estimated_audience_size_max: *int64*
- estimated_audience_size_min: *int64*
- languages: *array<str>*
- media_type: *enum {ALL, IMAGE, MEME, VIDEO, NONE}*
- publisher_platforms: *array<enum {FACEBOOK, INSTAGRAM, AUDIENCE_NETWORK, MESSENGER, WHATSAPP, OCULUS, THREADS}>*
- search_page_ids: *array<int64>*
- search_terms: *str* <span style="color:red">**(DEFAULT: "")**</span>
- search_type: *enum {KEYWORD_UNORDERED, KEYWORD_EXACT_PHRASE}* <span style="color:red">**(DEFAULT: KEYWORD_UNORDERED)**</span>
- unmask_removed_content: *bool* <span style="color:red">**(DEFAULT: False)**</span>

In [5]:
# Expected Payload
# It consists of a dict containing the access token and the desired parameters for the request
# Must have search_terms or search_page_ids parameters
# Please update the variable below with the desired search parameters
payload = {
            "access_token": ACCESS_TOKEN,
        }

#### Meta Ads API Available Fields

- All countries:
    * id
    * ad_creation_time
    * ad_creative_bodies
    * ad_creative_link_captions
    * ad_creative_link_descriptions
    * ad_creative_link_titles
    * ad_delivery_start_time
    * ad_delivery_stop_time
    * ad_snapshot_url
    * bylines
    * currency
    * delivery_by_region
    * demographic_distribution
    * estimated_audience_size
    * impressions
    * languages
    * page_id
    * page_name
    * publisher_platforms
    * spend
 
- Brazil only:
    * age_country_gender_reach_breakdown
    * br_total_reach
    * target_ages
    * target_gender
    * target_locations
    * total_reach_by_location
    
- EU only:
    * age_country_gender_reach_breakdown
    * beneficiary_payers
    * eu_total_reach
    * target_ages
    * target_gender
    * target_locations
    * total_reach_by_location


#### API Connection

In [6]:
# fields should be a string of selected fields separeted by comma, without any blank space
# e.g: "id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions"
fields = ""

In [7]:
# newest api version is v23.0 (08-09-2025)
api_url = f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"

In [8]:
# API Requests
def request_to_api(session, payload, api_url):
    response = session.get(api_url, params=payload)
    return response.json()

# Pagination
def paginate(session, payload, response):
    collected_ads = []
    while "paging" in response:
        next_page_url = response["paging"].get("next", "")
        sleep(1)
        response = request_to_api(session,payload, next_page_url)
        # The implementation of pagination ends here
        # Feel free to change implementation to extract data
        data = response.get("data", [])
        collected_ads.extend(data)
    return collected_ads

# Session
session = requests.Session()

In [9]:
# EU COUNTRIES ISO 3166
eu_countries_iso_list = ["AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "EL", "HU", "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT", "RO", "SK", "SI", "ES", "SE"]

### Special Criteria

**SC1: Does the platform provide an API to access its ad repository and extract data on advertising content?**

*This item verifies whether the platform provides an ad repository API with at least one endpoint for programmatically extracting advertising data. Full availability is confirmed when the API returns information on ads across all categories. The assessment should confirm that the endpoint allows the retrieval and storage of ad data without requiring privileged or internal access beyond standard developer registration.*


In [10]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET SEARCH TERM
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list, 
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": False
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data in response")
    print(response)

Got 25 ads in a single request


,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,languages,page_id,page_name,publisher_platforms,target_ages,target_gender,target_locations,total_reach_by_location,ad_creative_link_descriptions
0,468492669268944,2024-12-19,[Effecto.app helps people take control of ADHD...,[effecto.app],[Rewire Your ADHD Brain & Unlock Your Potential],2024-12-24,2024-12-28,"[{'country': 'PT', 'age_gender_breakdowns': [{...",[en],101597769428483,Effecto,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 192}]",NaN
1,540462482317039,2024-12-19,[Effecto.app helps people take control of ADHD...,[effecto.app],[Rewire Your ADHD Brain & Unlock Your Potential],2024-12-26,2024-12-28,"[{'country': 'SE', 'age_gender_breakdowns': [{...",[en],101597769428483,Effecto,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 4}]",NaN
2,927457872686945,2024-12-19,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[effecto.app],[Grab your plan for 51% OFF!],2024-12-23,2024-12-24,"[{'country': 'ES', 'age_gender_breakdowns': [{...",[en],351656754687509,Top 5 apps for ADHD,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 10002}]",NaN
3,1149767699841006,2024-12-19,[Effecto.app helps people take control of ADHD...,[effecto.app],[Rewire Your ADHD Brain & Unlock Your Potential],2024-12-23,2025-01-04,"[{'country': 'EE', 'age_gender_breakdowns': [{...",[en],101597769428483,Effecto,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 2229}]",NaN
4,1198154008406828,2024-12-19,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[effecto.app],[Grab your plan for 51% OFF!],2024-12-19,2024-12-21,"[{'country': 'EE', 'age_gender_breakdowns': [{...",[en],351656754687509,Top 5 apps for ADHD,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 5213}]",NaN
5,1645469656179516,2024-12-19,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[effecto.app],[Grab your plan for 51% OFF!],2024-12-25,2024-12-27,"[{'country': 'IT', 'age_gender_breakdowns': [{...",[en],351656754687509,Top 5 apps for ADHD,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 26032}]",NaN
6,1681368622408808,2024-12-19,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[effecto.app],[Grab your plan for 51% OFF!],2024-12-26,2024-12-28,"[{'country': 'FR', 'age_gender_breakdowns': [{...",[en],351656754687509,Top 5 apps for ADHD,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 8067}]",NaN
7,1699924333908780,2024-12-19,[Effecto.app helps people take control of ADHD...,[effecto.app],[Rewire Your ADHD Brain & Unlock Your Potential],2024-12-19,2025-01-01,"[{'country': 'RO', 'age_gender_breakdowns': [{...",[en],101597769428483,Effecto,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 74}]",NaN
8,1990654091358510,2024-12-19,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[effecto.app],[Grab your plan for 51% OFF!],2024-12-24,2024-12-25,"[{'country': 'CY', 'age_gender_breakdowns': [{...",[en],351656754687509,Top 5 apps for ADHD,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 6337}]",NaN
9,573867262268518,2024-12-19,[✨️️️️️️️ Stop overthinking and start living t...,[mindway.app],[Chronic Overthi

In [11]:
# Checking type of ad in response
political_fields = ["bylines", "currency", "delivery_by_region", "demographic_distribution", "estimated_audience_size", "impressions", "spend"]
political_ads = 0
other_ads = 0
for ad in response["data"]:
    has_political_fields = False
    for k in ad.keys():
        if k in political_fields:
            has_political_fields = True
            break
    if has_political_fields:
        political_ads += 1
    else:
        other_ads += 1

print("Total political ads:", political_ads)
print("Total other categories ads:", other_ads)

Total political ads: 0
Total other categories ads: 25


In [12]:
# EMPLOYMENT_ADS
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,page_id,page_name,publisher_platforms"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET SEARCH TERM
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "EMPLOYMENT_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in EMPLOYMENT_ADS in a single request")
    print(response)
else:
    print("No data found in EMPLOYMENT_ADS")
    print(response)

Got 25 in EMPLOYMENT_ADS in a single request
{'data': [{'id': '823490390655448', 'ad_creation_time': '2025-11-20', 'ad_creative_bodies': ['Discover this opportunity! 🔥'], 'ad_creative_link_captions': ['bestagencywork.com'], 'ad_creative_link_titles': ['Discover this opportunity! 🔥'], 'ad_delivery_start_time': '2025-11-21', 'ad_delivery_stop_time': '2025-11-21', 'page_id': '615127158344887', 'page_name': 'Curlia Haeper', 'publisher_platforms': ['facebook', 'instagram']}, {'id': '838074162485111', 'ad_creation_time': '2025-11-20', 'ad_creative_bodies': ['Pest control technicians can earn industry certifications in specialized areas like termite treatment, wildlife management, and fumigation 🐛'], 'ad_creative_link_captions': ['hometalk.com'], 'ad_creative_link_descriptions': ['The pest control industry plays a crucial role in public health and safety, protecting homes and businesses from the threats posed by pests. With increasing awareness about the importance of pest management, this se

In [13]:
# FINANCIAL_PRODUCTS_AND_SERVICES_ADS
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,page_id,page_name,publisher_platforms"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET SEARCH TERM
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "FINANCIAL_PRODUCTS_AND_SERVICES_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in FINANCIAL_PRODUCTS_AND_SERVICES_ADS in a single request")
    print(response)
else:
    print("No data found in FINANCIAL_PRODUCTS_AND_SERVICES_ADS")
    print(response)

Got 25 in FINANCIAL_PRODUCTS_AND_SERVICES_ADS in a single request
{'data': [{'id': '1176841204576576', 'ad_creation_time': '2025-11-20', 'ad_creative_bodies': ['Få enestående dansk service og ekspertise til din internationale helbredsforsikring, både i og udenfor Danmark. Beskyt dit helbred over hele verden.', 'Få enestående dansk service og ekspertise til din internationale helbredsforsikring, både i og udenfor Danmark. Beskyt dit helbred over hele verden.'], 'ad_creative_link_captions': ['healthinsuranceinstantly.com', 'healthinsuranceinstantly.com'], 'ad_creative_link_descriptions': ['En sundhedsforsikring der forstår dig', 'En sundhedsforsikring der forstår dig'], 'ad_creative_link_titles': ['Perfekt til Danskere I Udlandet', 'Perfekt til Danskere I Udlandet'], 'ad_delivery_start_time': '2025-11-20', 'page_id': '110817364130442', 'page_name': 'Health Insurance Instantly', 'publisher_platforms': ['facebook', 'instagram', 'audience_network', 'messenger', 'threads']}, {'id': '13836000

In [14]:
# HOUSING_ADS
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,page_id,page_name,publisher_platforms"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "house" # SET SEARCH TERM
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "HOUSING_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in HOUSING_ADS in a single request")
    print(response)
else:
    print("No data found in HOUSING_ADS")
    print(response)

Got 25 in HOUSING_ADS in a single request
{'data': [{'id': '854441957228489', 'ad_creation_time': '2025-11-20', 'ad_creative_bodies': ['Global loan OFW International Bank\nWE ARE OPEN FOR:\n📌BUSINESS LOAN\n📌PERSONAL LOAN\n📌OFW LOAN\n📌EMERGENCY LOAN\n📌HOUSE LOAN\n📌CASH LOAN\n📌STUDENTS LOAN\n📌URGENT LOAN\n📌TUITION FEE LOAN\nAVAILABLE CURRENCY\n$, ₪, ₱, ¥, €, 원, ₹, PKR....\nLEGIT LOAN: PHP 100K UP - 3M'], 'ad_creative_link_captions': ['fb.com'], 'ad_creative_link_titles': ['0.3% Monthly fixed interest'], 'ad_delivery_start_time': '2025-11-20', 'page_id': '662302426973350', 'page_name': 'Rejoy Jose Banas', 'publisher_platforms': ['facebook', 'messenger']}, {'id': '4042275692663970', 'ad_creation_time': '2025-11-20', 'ad_creative_bodies': ['Bara 40 minuter från centrala Málaga representerar denna lantliga egendom en investeringsmöjlighet. Med direkt tillgång från motorvägarna A-45 och AP-46 omfattar fastigheten 47 hektar förstklassig odlingsmark, mycket lämplig för vinodling, olivträd och m

In [15]:
# POLITICAL_AND_ISSUE_ADS
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET SEARCH TERM
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in POLITICAL_AND_ISSUE_ADS in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found in POLITICAL_AND_ISSUE_ADS")
    print(response)

No data found in POLITICAL_AND_ISSUE_ADS
{'error': {'message': "(#100) Political ad searches for countries in the European Union aren't available.", 'type': 'OAuthException', 'code': 100, 'fbtrace_id': 'AKKAUSWOIT-qMImMX2x36fo'}}


In [16]:
# POLITICAL_AND_ISSUE_ADS
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET SEARCH TERM
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ['NL'],  # Testing with a single country
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in POLITICAL_AND_ISSUE_ADS in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found in POLITICAL_AND_ISSUE_ADS")
    print(response)

No data found in POLITICAL_AND_ISSUE_ADS
{'error': {'message': "(#100) Political ad searches for countries in the European Union aren't available.", 'type': 'OAuthException', 'code': 100, 'fbtrace_id': 'AMYIFMA-KYa_WGemxUWflMg'}}


**SC3: Can data from both active and inactive ads be extracted?**

*This item verifies whether the platform allows the extraction of ad data through either the GUI or the API, from at least one day after publication to at least one year prior. Full availability is confirmed when both active and inactive ad data are delivered across all ad categories. The assessment should test the interface and endpoints to confirm whether both active and inactive ads can be retrieved.*


In [17]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET SEARCH TERM
active_payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ACTIVE",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
inactive_payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "INACTIVE",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
active_response = request_to_api(session, active_payload, api_url)
inactive_response = request_to_api(session, inactive_payload, api_url)
if "data" in active_response:
    print(f"Got {len(active_response['data'])} ACTIVE ads in a single request")
    active_df = pd.DataFrame(active_response["data"])
    display(active_df)
else:
    print("No ACTIVE ads were found")
    print(active_response)
if "data" in inactive_response:
    print(f"Got {len(inactive_response['data'])} INACTIVE ads in a single request")
    inactive_df = pd.DataFrame(inactive_response["data"])
    display(inactive_df)
else:
    print("No INACTIVE ads were found")
    print(inactive_response)


Got 25 ACTIVE ads in a single request


,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,age_country_gender_reach_breakdown,languages,page_id,page_name,publisher_platforms,target_ages,target_gender,target_locations,total_reach_by_location
0,1444419930160485,2025-09-02,"[Get better flow, improved adaptation, reduced...",[info.engage.solventum.com],[Request a demo page],[Request a composite warming demo],2025-11-18,"[{'country': 'IE', 'age_gender_breakdowns': [{...",[en],350641159250,Solventum Dental,"[facebook, instagram, audience_network, threads]","[24, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 278}, {'key': 'GB', 'v..."
1,847056504396966,2025-10-22,[Waarom reageren sommige lichamen op een zacht...,[holisleren.nl],[👉],[De spiertest-revolutie die de holistische gen...,2025-10-23,"[{'country': 'BE', 'age_gender_breakdowns': [{...",[nl],106398342445417,Holisleren,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Worldwide', 'num_obfuscated': 0, 't...","[{'key': 'EU', 'value': 27964}, {'key': 'GB', ..."
2,1137687685158875,2025-10-22,[Waarom reageren sommige lichamen op een zacht...,[holisleren.nl],[👉],[De spiertest-revolutie die de holistische gen...,2025-10-23,"[{'country': 'PT', 'age_gender_breakdowns': [{...",[nl],106398342445417,Holisleren,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Worldwide', 'num_obfuscated': 0, 't...","[{'key': 'EU', 'value': 17989}, {'key': 'GB', ..."
3,1156818339116178,2025-10-22,[Waarom reageren sommige lichamen op een zacht...,[holisleren.nl],[👉],[De spiertest-revolutie die de holistische gen...,2025-10-23,"[{'country': 'BE', 'age_gender_breakdowns': [{...",[nl],106398342445417,Holisleren,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Worldwide', 'num_obfuscated': 0, 't...","[{'key': 'EU', 'value': 100918}, {'key': 'GB',..."
4,1919099282369658,2025-10-22,[Waarom reageren sommige lichamen op een zacht...,[holisleren.nl],[👉],[De spiertest-revolutie die de holistische gen...,2025-10-23,"[{'country': 'BE', 'age_gender_breakdowns': [{...",[nl],106398342445417,Holisleren,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Worldwide', 'num_obfuscated': 0, 't...","[{'key': 'EU', 'value': 3730}, {'key': 'GB', '..."
5,823468013933436,2025-11-21,[Penny and Mason are childhood best friends wh...,[htgq3.com],NaN,[How do I hide my cancer from the boy I love?],2025-11-21,"[{'country': 'SK', 'age_gender_breakdowns': [{...",[en],635889216282246,ShortStories Club,"[facebook, instagram, audience_network, messen...","[18, 44]",All,"[{'name': 'Worldwide', 'num_obfuscated': 0, 't...","[{'key': 'EU', 'value': 4}, {'key': 'GB', 'val..."
6,1144985784035628,2025-11-21,[Penny and Mason are childhood best friends wh...,[htgq3.com],NaN,[How do I hide my cancer from the boy I love?],2025-11-21,"[{'country': 'AT', 'age_gender_breakdowns': [{...",[en],635889216282246,ShortStories Club,"[facebook, instagram, audience_network, messen...","[18, 44]",All,"[{'name': 'Worldwide', 'num_obfuscated': 0, 't...","[{'key': 'EU', 'value': 3}, {'key': 'GB', 'val..."
7,780613925012582,2025-11-21,[Penny and Mason are childhood best friends wh...,[htgq3.com],NaN,[How do I hide my cancer from the boy I love?],2025-11-21,"[{'country': 'AT', 'age_gender_breakdowns': [{...",[en],104166679290552,Reelshort-Video stories,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Worldwide', 'num_obfuscated': 0, 't...","[{'key': 'EU', 'value': 30}, {'key': 'GB', 'va..."
8,852590087218124,2025-11-21,[Penny and Mason are childhood best friends wh...,[htgq3.com],NaN,[How do I hide my cancer from the boy I love?],2025-11-21,"[{'country': 'DK', 'age_gender_breakdowns': [{...",[en],104166679290552,Reelshort-Video stories,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Worldwide', 'num_obfuscated'

Got 25 INACTIVE ads in a single request


,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,languages,page_id,page_name,publisher_platforms,target_ages,target_gender,target_locations,total_reach_by_location,ad_creative_link_descriptions
0,468492669268944,2024-12-19,[Effecto.app helps people take control of ADHD...,[effecto.app],[Rewire Your ADHD Brain & Unlock Your Potential],2024-12-24,2024-12-28,"[{'country': 'PT', 'age_gender_breakdowns': [{...",[en],101597769428483,Effecto,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 192}]",NaN
1,540462482317039,2024-12-19,[Effecto.app helps people take control of ADHD...,[effecto.app],[Rewire Your ADHD Brain & Unlock Your Potential],2024-12-26,2024-12-28,"[{'country': 'SE', 'age_gender_breakdowns': [{...",[en],101597769428483,Effecto,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 4}]",NaN
2,927457872686945,2024-12-19,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[effecto.app],[Grab your plan for 51% OFF!],2024-12-23,2024-12-24,"[{'country': 'ES', 'age_gender_breakdowns': [{...",[en],351656754687509,Top 5 apps for ADHD,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 10002}]",NaN
3,1149767699841006,2024-12-19,[Effecto.app helps people take control of ADHD...,[effecto.app],[Rewire Your ADHD Brain & Unlock Your Potential],2024-12-23,2025-01-04,"[{'country': 'EE', 'age_gender_breakdowns': [{...",[en],101597769428483,Effecto,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 2229}]",NaN
4,1198154008406828,2024-12-19,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[effecto.app],[Grab your plan for 51% OFF!],2024-12-19,2024-12-21,"[{'country': 'EE', 'age_gender_breakdowns': [{...",[en],351656754687509,Top 5 apps for ADHD,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 5213}]",NaN
5,1645469656179516,2024-12-19,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[effecto.app],[Grab your plan for 51% OFF!],2024-12-25,2024-12-27,"[{'country': 'IT', 'age_gender_breakdowns': [{...",[en],351656754687509,Top 5 apps for ADHD,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 26032}]",NaN
6,1681368622408808,2024-12-19,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[effecto.app],[Grab your plan for 51% OFF!],2024-12-26,2024-12-28,"[{'country': 'FR', 'age_gender_breakdowns': [{...",[en],351656754687509,Top 5 apps for ADHD,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 8067}]",NaN
7,1699924333908780,2024-12-19,[Effecto.app helps people take control of ADHD...,[effecto.app],[Rewire Your ADHD Brain & Unlock Your Potential],2024-12-19,2025-01-01,"[{'country': 'RO', 'age_gender_breakdowns': [{...",[en],101597769428483,Effecto,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 74}]",NaN
8,1990654091358510,2024-12-19,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[effecto.app],[Grab your plan for 51% OFF!],2024-12-24,2024-12-25,"[{'country': 'CY', 'age_gender_breakdowns': [{...",[en],351656754687509,Top 5 apps for ADHD,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 6337}]",NaN
9,573867262268518,2024-12-19,[✨️️️️️️️ Stop overthinking and start living t...,[mindway.app],[Chronic Overthi

### ACCESSIBILITY


**OC3: Can the requested data be extracted directly from the ad repository response?**

This item verifies whether the platform ad repository returns structured data on ad creatives and authorship directly in the response, rather than providing a link that redirects to the data. Audiovisual media files and data (e.g., images, videos, and audio) should not be considered when assessing this item. The assessment should examine sample data responses from both the ad repository GUI and API to confirm that the requested public data is included in the returned payload.*


In [18]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,ad_creative_bodies,ad_creative_link_descriptions,ad_creative_link_titles,bylines,page_id,page_name"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET SEARCH TERM
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No ads were found")
    print(response)

Got 25 ads in a single request


,id,ad_creative_bodies,ad_creative_link_titles,page_id,page_name,ad_creative_link_descriptions
0,468492669268944,[Effecto.app helps people take control of ADHD...,[Rewire Your ADHD Brain & Unlock Your Potential],101597769428483,Effecto,NaN
1,540462482317039,[Effecto.app helps people take control of ADHD...,[Rewire Your ADHD Brain & Unlock Your Potential],101597769428483,Effecto,NaN
2,927457872686945,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[Grab your plan for 51% OFF!],351656754687509,Top 5 apps for ADHD,NaN
3,1149767699841006,[Effecto.app helps people take control of ADHD...,[Rewire Your ADHD Brain & Unlock Your Potential],101597769428483,Effecto,NaN
4,1198154008406828,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[Grab your plan for 51% OFF!],351656754687509,Top 5 apps for ADHD,NaN
5,1645469656179516,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[Grab your plan for 51% OFF!],351656754687509,Top 5 apps for ADHD,NaN
6,1681368622408808,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[Grab your plan for 51% OFF!],351656754687509,Top 5 apps for ADHD,NaN
7,1699924333908780,[Effecto.app helps people take control of ADHD...,[Rewire Your ADHD Brain & Unlock Your Potential],101597769428483,Effecto,NaN
8,1990654091358510,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[Grab your plan for 51% OFF!],351656754687509,Top 5 apps for ADHD,NaN
9,573867262268518,[✨️️️️️️️ Stop overthinking and start living t...,[Chronic Overthinking Test 👉],214946655027813,Your Personal Coach Sophia,NaN


In [19]:
if response["data"]:
    URL_PATTERN = r"https?://\S+"
    df = pd.DataFrame(response["data"])
    schema = pa.DataFrameSchema(
        columns={},
        checks=pa.Check(
            lambda df: (
                df.select_dtypes(include=['object'])
                .apply(lambda s: s.str.contains(URL_PATTERN).any(), axis=0)
                .sum() == 0
            ),
            name="DirectResponse"
        ),
        strict=False 
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: No links found in any ad creative or authorship column.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Links are present in ad creatives or authorship columns.")
        print(err)

✅ Validation Successful: No links found in any ad creative or authorship column.


**OC5: Can data from an individual ad be retrieved from the platform?**

*This item verifies whether it is possible to retrieve data from a specific advertisement on the ad repository using a unique identifier, rather than relying on search terms or other parameters and filters. The assessment should review the ad repository documentation and test available features to confirm that an individual ad can be retrieved directly by its unique identifier.*



Ad Library link: [AD ID](https://www.facebook.com/ads/library/?id=468492669268944) PLEASE CHANGE THE AD

In [20]:
# Use this cell to develop an example where a request for ads from a campaign is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
ad_id = '468492669268944' # SET VALUE HERE
fields="id"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": ad_id,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    found_ad = False
    for ad in response["data"]:
        if ad["id"] == ad_id:
            found_ad = True
            break
    if found_ad:
        print(f"Found ad {ad_id} in response")
        print(response)
    else:
        print(f"Ad {ad_id} not found in response")
        print(response)
else:
    print("No data were found in response")
    print(response)

Ad 468492669268944 not found in response
{'data': []}


In [21]:
ad_id = '468492669268944' # SET AD ID
fields="id"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["GB"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_page_ids": ad_id,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    found_ad = False
    for ad in response["data"]:
        if ad["id"] == ad_id:
            found_ad = True
            break
    if found_ad:
        print(f"Found ad {ad_id} in response")
        print(response)
    else:
        print(f"Ad {ad_id} not found in response")
        print(response)
else:
    print("No data were found in response")
    print(response)

Ad 468492669268944 not found in response
{'data': []}


**OC6: Can data from ads served by a specific advertiser be retrieved from the platform?**

*This item verifies whether it is possible to retrieve data from ads run by a specific advertiser, via their username or unique identifier. The assessment should review the ad repository documentation and test any available feature to retrieve data from an individual advertiser.*


In [22]:
# Use this cell to develop an example where a request for ads from an advertiser is made.
# Please leave the result as the output of this cell.
advertiser = '101597769428483' # SET VALUE
fields="id,page_id,page_name"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_page_ids": advertiser,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    found_advertiser = False
    other_advertisers = 0
    for ad in response["data"]:
        if ad["page_id"] == advertiser:
            found_advertiser = True
        else:
            other_advertisers += 1
    if found_advertiser and other_advertisers == 0:
        print(f"Only ads from advertiser {advertiser} were found in response")
    elif found_advertiser and other_advertisers > 0:
        print(f"Found ads from advertiser {advertiser} and {other_advertisers} others in response")
    else:
        print(f"Advertiser {advertiser} not found in response")
else:
    print("No data were found in response")
    print(response)

Only ads from advertiser 101597769428483 were found in response


**OC7: Can ad data be retrieved from the platform using search terms?**

*This item verifies whether data on ad campaigns can be retrieved via individual or combined search terms, enabling the creation of a dataset composed of ads that mention those terms. The assessment should test search-related features to confirm that queries using keywords return relevant ad campaign data.*


In [23]:
# Use this cell to develop an example where a request for ads is made using a search term.
# Please leave the result as the output of this cell.
search_term = "health" # SET THE VALUE HERE
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": search_term,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request using the search term {search_term}")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No ads were found")
    print(response)

Got 25 ads in a single request using the search term health


,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,languages,page_id,page_name,publisher_platforms,target_ages,target_gender,target_locations,total_reach_by_location,ad_creative_link_descriptions
0,468492669268944,2024-12-19,[Effecto.app helps people take control of ADHD...,[effecto.app],[Rewire Your ADHD Brain & Unlock Your Potential],2024-12-24,2024-12-28,"[{'country': 'PT', 'age_gender_breakdowns': [{...",[en],101597769428483,Effecto,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 192}]",NaN
1,540462482317039,2024-12-19,[Effecto.app helps people take control of ADHD...,[effecto.app],[Rewire Your ADHD Brain & Unlock Your Potential],2024-12-26,2024-12-28,"[{'country': 'SE', 'age_gender_breakdowns': [{...",[en],101597769428483,Effecto,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 4}]",NaN
2,927457872686945,2024-12-19,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[effecto.app],[Grab your plan for 51% OFF!],2024-12-23,2024-12-24,"[{'country': 'ES', 'age_gender_breakdowns': [{...",[en],351656754687509,Top 5 apps for ADHD,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 10002}]",NaN
3,1149767699841006,2024-12-19,[Effecto.app helps people take control of ADHD...,[effecto.app],[Rewire Your ADHD Brain & Unlock Your Potential],2024-12-23,2025-01-04,"[{'country': 'EE', 'age_gender_breakdowns': [{...",[en],101597769428483,Effecto,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 2229}]",NaN
4,1198154008406828,2024-12-19,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[effecto.app],[Grab your plan for 51% OFF!],2024-12-19,2024-12-21,"[{'country': 'EE', 'age_gender_breakdowns': [{...",[en],351656754687509,Top 5 apps for ADHD,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 5213}]",NaN
5,1645469656179516,2024-12-19,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[effecto.app],[Grab your plan for 51% OFF!],2024-12-25,2024-12-27,"[{'country': 'IT', 'age_gender_breakdowns': [{...",[en],351656754687509,Top 5 apps for ADHD,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 26032}]",NaN
6,1681368622408808,2024-12-19,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[effecto.app],[Grab your plan for 51% OFF!],2024-12-26,2024-12-28,"[{'country': 'FR', 'age_gender_breakdowns': [{...",[en],351656754687509,Top 5 apps for ADHD,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 8067}]",NaN
7,1699924333908780,2024-12-19,[Effecto.app helps people take control of ADHD...,[effecto.app],[Rewire Your ADHD Brain & Unlock Your Potential],2024-12-19,2025-01-01,"[{'country': 'RO', 'age_gender_breakdowns': [{...",[en],101597769428483,Effecto,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 74}]",NaN
8,1990654091358510,2024-12-19,[⭐⭐⭐⭐⭐\nLike Flo but for ADHD\n🌱 Find your ult...,[effecto.app],[Grab your plan for 51% OFF!],2024-12-24,2024-12-25,"[{'country': 'CY', 'age_gender_breakdowns': [{...",[en],351656754687509,Top 5 apps for ADHD,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 6337}]",NaN
9,573867262268518,2024-12-19,[✨️️️️️️️ Stop overthinking and start living t...,[mindway.app],[Chronic Overthi

**OC8: Does the platform use locale-neutral data representations?**

*This item verifies whether locale-sensitive data (e.g., timestamps, currency, numbers) are provided in a locale-neutral format, or, if that is not possible, whether relevant locale metadata is included. The assessment should review the ad repository documentation and inspect sample responses to confirm the presence of standardized formats or accompanying metadata.*


In [26]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
query = "health" # SET VALUE
fields="id,ad_creation_time,ad_delivery_start_time,ad_delivery_stop_time,currency"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No ads were found")
    print(response)

Got 25 ads in a single request


,id,ad_creation_time,ad_delivery_start_time,ad_delivery_stop_time
0,468492669268944,2024-12-19,2024-12-24,2024-12-28
1,540462482317039,2024-12-19,2024-12-26,2024-12-28
2,927457872686945,2024-12-19,2024-12-23,2024-12-24
3,1149767699841006,2024-12-19,2024-12-23,2025-01-04
4,1198154008406828,2024-12-19,2024-12-19,2024-12-21
5,1645469656179516,2024-12-19,2024-12-25,2024-12-27
6,1681368622408808,2024-12-19,2024-12-26,2024-12-28
7,1699924333908780,2024-12-19,2024-12-19,2025-01-01
8,1990654091358510,2024-12-19,2024-12-24,2024-12-25
9,573867262268518,2024-12-19,2024-12-23,2025-02-22


In [25]:
if "data" in response:
    df = pd.json_normalize(response["data"])
    df["ad_creation_time"] = pd.to_datetime( df["ad_creation_time"])
    df["ad_delivery_start_time"] = pd.to_datetime( df["ad_delivery_start_time"])
    df["ad_delivery_stop_time"] = pd.to_datetime( df["ad_delivery_stop_time"])
    schema = pa.DataFrameSchema(
        {
            "ad_creation_time": pa.Column(
                pa.DateTime, 
                nullable=True, 
                title="Ad creation time"
            ),
            "ad_delivery_start_time": pa.Column(
                pa.DateTime, 
                nullable=False, 
                title="Ad delivery start time"
            ),
            "ad_delivery_stop_time": pa.Column(
                pa.DateTime, 
                nullable=True, 
                title="Ad delivery stop time"
            ),
            "currency": pa.Column(
                pa.String,
                pa.Check.isin(["EUR", "SEK", "BGN", "CZK", "DKK", "HUF", "PLN", "RON"]), 
                nullable=False, 
                title="Currency"
            ),
        
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: Data is locale-neutral represented.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Data is not locale-neutral represented.")
        print(err)

❌ Validation Failed: Data is not locale-neutral represented.
{
    "SCHEMA": {
        "COLUMN_NOT_IN_DATAFRAME": [
            {
                "schema": null,
                "column": null,
                "check": "column_in_dataframe",
                "error": "column 'currency' not in dataframe. Columns in dataframe: ['id', 'ad_creation_time', 'ad_delivery_start_time', 'ad_delivery_stop_time']"
            }
        ]
    }
}


### COMPLETENESS

**OC9: Does the platform provide data that allows the identification of advertisers who ran ads?**

*This item verifies whether the platform discloses information on the advertisers responsible for the identified ads. The assessment should confirm whether the advertiser’s page name, URL, and unique identifier can be retrieved.*



In [27]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
query = "*" # SET VALUE
fields="id,page_id,page_name"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No ads were found")
    print(response)

Got 25 ads in a single request


,id,page_id,page_name
0,1200639028612833,699930669874195,Joyreels-4-jin4
1,1876722389910719,699930669874195,Joyreels-4-jin4
2,683037654351925,635626916309545,Master Lee
3,753234767697431,635626916309545,Master Lee
4,854475540339910,635626916309545,Master Lee
5,1553320475853426,635626916309545,Master Lee
6,3654874951487455,635626916309545,Master Lee
7,4236116543318552,172184559301717,City of Hunt-Ⅱ
8,1398240104596751,189764851743163,Kelly Shutt
9,1679127252750216,189764851743163,Kelly Shutt


In [28]:
if "data" in response:
    df = pd.DataFrame(response["data"])
    schema = pa.DataFrameSchema(
        {
            "page_id": pa.Column(
                pa.String, 
                nullable=False,  
                title="Page Id"
            ),
            "page_name": pa.Column(
                pa.String, 
                nullable=False,   
                title="Page Name"
            ),
        
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: Advertisers information columns are present and non-null.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Missing advertisers information.")
        print(err)

✅ Validation Successful: Advertisers information columns are present and non-null.


**OC10: Does the platform provide data on the funders who paid for ads?**

*This item verifies whether the platform provides data on who paid for ad campaigns. The assessment should confirm whether any sponsor information can be retrieved.*


In [29]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
query = "health" # SET VALUE
fields="id,bylines"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": False
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No ads were found")
    print(response)

Got 25 ads in a single request


,id
0,468492669268944
1,540462482317039
2,927457872686945
3,1149767699841006
4,1198154008406828
5,1645469656179516
6,1681368622408808
7,1699924333908780
8,1990654091358510
9,573867262268518


In [30]:
if "data" in response:
    df = pd.DataFrame(response["data"])
    schema = pa.DataFrameSchema(
        {
            "bylines": pa.Column(
                pa.String, 
                nullable=False,  
                title="Bylines"
            )
        
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: Funders information columns are present and non-null.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Missing funders information.")
        print(err)

❌ Validation Failed: Missing funders information.
{
    "SCHEMA": {
        "COLUMN_NOT_IN_DATAFRAME": [
            {
                "schema": null,
                "column": null,
                "check": "column_in_dataframe",
                "error": "column 'bylines' not in dataframe. Columns in dataframe: ['id']"
            }
        ]
    }
}


**OC11: Does the platform provide data on the period during which ads were served?**

*This item verifies whether the platform provides data on the days on which ad campaigns ran. The assessment should review ad records to confirm that campaigns include start and end dates (or equivalent temporal markers) covering their active period*

In [31]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
query = "health" # SET VALUE
fields="id,ad_delivery_start_time,ad_delivery_stop_time"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "INACTIVE",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No ads were found")
    print(response)

Got 25 ads in a single request


,id,ad_delivery_start_time,ad_delivery_stop_time
0,468492669268944,2024-12-24,2024-12-28
1,540462482317039,2024-12-26,2024-12-28
2,927457872686945,2024-12-23,2024-12-24
3,1149767699841006,2024-12-23,2025-01-04
4,1198154008406828,2024-12-19,2024-12-21
5,1645469656179516,2024-12-25,2024-12-27
6,1681368622408808,2024-12-26,2024-12-28
7,1699924333908780,2024-12-19,2025-01-01
8,1990654091358510,2024-12-24,2024-12-25
9,573867262268518,2024-12-23,2025-02-22


In [32]:
if "data" in response:
    df = pd.DataFrame(response["data"])
    df["ad_delivery_start_time"] = pd.to_datetime(df["ad_delivery_start_time"])
    df["ad_delivery_stop_time"] = pd.to_datetime(df["ad_delivery_stop_time"])
    schema = pa.DataFrameSchema(
        {
            "ad_delivery_start_time": pa.Column(
                pa.DateTime, 
                nullable=False,  
                title="Ad delivery start time"
            ),
            "ad_delivery_stop_time": pa.Column(
                pa.DateTime, 
                nullable=False,  
                title="Ad delivery stop time"
            )
        
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: Ads' run period dates are present and non-null.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Missing ads' run period dates.")
        print(err)


✅ Validation Successful: Ads' run period dates are present and non-null.


**OC12: Does the platform provide data on user engagement with ads?**

*This item verifies whether the platform provides data on the total number of user interactions with ad campaigns (e.g., likes, comments, shares, clicks). The assessment should review ad records to confirm that engagement metrics are available and clearly linked to each campaign.*


There's no field to retrieve user interactions. A complete list of available fields can be found in: https://developers.facebook.com/docs/graph-api/reference/archived-ad/

**OC13: Does the platform indicate whether ads were placed by verified or unverified advertisers?**

*This item verifies whether the platform clearly indicates whether advertisers were verified at the time their ads were served. The assessment should review ad records to confirm that a verification status field is present.*


There's no field that indicates if a advertiser is verified. A complete list of available fields can be found in: https://developers.facebook.com/docs/graph-api/reference/archived-ad/

### COMPLIANCE

**OC14: Does the platform flag ads that were removed due to violations of its guidelines or relevant legislation?**

*This item verifies whether the platform indicates when an ad has been moderated. At a minimum, the platform should provide the reason for removal and the date. The assessment should review ad records to confirm that moderated ads are flagged and that the corresponding moderation details are clearly documented.*



In [35]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
query = "*" # SET VALUE
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            'search_page_ids' : ['189764851743163'],
            "search_terms": query,
            "unmask_removed_content": True # unmask set to TRUE
        }
response = request_to_api(session, payload, api_url)
df = pd.DataFrame(response["data"])
display(df)

,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,languages,page_id,page_name,publisher_platforms,target_ages,target_gender,target_locations,total_reach_by_location
0,1398240104596751,2025-08-17,[🌟 Say Goodbye to Neuropathy Pain with Our Neu...,[healthhubzz.com],[🌟 Rapid Pain Relief: Quickly soothe nerve pai...,[🌟 Instant Relief with Our Nerve Healing Cream...,2025-08-17,2025-08-18,"[{'country': 'BE', 'age_gender_breakdowns': [{...",[en],189764851743163,Kelly Shutt,[facebook],"[18, 65]",All,"[{'name': 'European Economic Area (EEA)', 'num...","[{'key': 'EU', 'value': 746}, {'key': 'GB', 'v..."
1,1679127252750216,2025-08-17,[🌟 Say Goodbye to Neuropathy Pain with Our Neu...,[healthhubzz.com],[🌟 Rapid Pain Relief: Quickly soothe nerve pai...,[🌟 Instant Relief with Our Nerve Healing Cream...,2025-08-17,2025-08-19,"[{'country': 'CY', 'age_gender_breakdowns': [{...",[en],189764851743163,Kelly Shutt,[facebook],"[18, 65]",All,"[{'name': 'European Economic Area (EEA)', 'num...","[{'key': 'EU', 'value': 892}, {'key': 'GB', 'v..."
2,1775545670017730,2025-08-17,[🌟 Say Goodbye to Neuropathy Pain with Our Neu...,[healthhubzz.com],[🌟 Rapid Pain Relief: Quickly soothe nerve pai...,[🌟 Instant Relief with Our Nerve Healing Cream...,2025-08-17,2025-08-18,"[{'country': 'IE', 'age_gender_breakdowns': [{...",[en],189764851743163,Kelly Shutt,[facebook],"[18, 65]",All,"[{'name': 'European Economic Area (EEA)', 'num...","[{'key': 'EU', 'value': 184}, {'key': 'GB', 'v..."
3,3699250353711993,2025-08-17,[🌟 Say Goodbye to Neuropathy Pain with Our Neu...,[healthhubzz.com],[🌟 Rapid Pain Relief: Quickly soothe nerve pai...,[🌟 Instant Relief with Our Nerve Healing Cream...,2025-08-17,2025-08-18,"[{'country': 'PL', 'age_gender_breakdowns': [{...",[en],189764851743163,Kelly Shutt,[facebook],"[18, 65]",All,"[{'name': 'European Economic Area (EEA)', 'num...","[{'key': 'EU', 'value': 121}, {'key': 'GB', 'v..."
4,23867551009608298,2025-08-17,[🌟 Say Goodbye to Neuropathy Pain with Our Neu...,[healthhubzz.com],[🌟 Rapid Pain Relief: Quickly soothe nerve pai...,[🌟 Instant Relief with Our Nerve Healing Cream...,2025-08-17,2025-08-18,"[{'country': 'DE', 'age_gender_breakdowns': [{...",[en],189764851743163,Kelly Shutt,[facebook],"[18, 65]",All,"[{'name': 'European Economic Area (EEA)', 'num...","[{'key': 'EU', 'value': 177}, {'key': 'GB', 'v..."
5,739620682176679,2025-08-17,[🌟 Say Goodbye to Neuropathy Pain with Our Neu...,[healthhubzz.com],[🌟 Rapid Pain Relief: Quickly soothe nerve pai...,[🌟 Instant Relief with Our Nerve Healing Cream...,2025-08-17,2025-08-27,"[{'country': 'AT', 'age_gender_breakdowns': [{...",[en],189764851743163,Kelly Shutt,[facebook],"[18, 65]",All,"[{'name': 'European Economic Area (EEA)', 'num...","[{'key': 'EU', 'value': 28069}, {'key': 'GB', ..."
6,777150081484411,2025-08-17,[🌟 Say Goodbye to Neuropathy Pain with Our Neu...,[healthhubzz.com],[🌟 Rapid Pain Relief: Quickly soothe nerve pai...,[🌟 Instant Relief with Our Nerve Healing Cream...,2025-08-17,2025-08-27,"[{'country': 'AT', 'age_gender_breakdowns': [{...",[en],189764851743163,Kelly Shutt,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'European Economic Area (EEA)', 'num...","[{'key': 'EU', 'value': 9985}, {'key': 'GB', '..."
7,1462225944816126,2025-08-17,[🌟 Say Goodbye to Neuropathy Pain with Our Neu...,[healthhubzz.com],[🌟 Rapid Pain Relief: Quickly soothe nerve pai...,[🌟 Instant Relief with Our Nerve Healing Cream...,2025-08-17,2025-08-27,"[{'country': 'AT', 'age_gender_breakdowns': [{...",[en],189764851743163,Kelly Shutt,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'European Economic Area (EEA)', 'num...","[{'key': 'EU', 'value': 14291}, {'key': 'GB', ..."
8,2269876786796552,2025-08-17,[🌟 Say Goodbye to Neuropathy Pain with Our Neu...,[healthhubzz.com],[🌟 Rapid Pain Relief: Quic

In [ ]:
query = "*" # SET VALUE
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            'search_page_ids' : ['189764851743163'],
            "search_terms": query,
            "unmask_removed_content": False # unmask set to FALSE
        }
response = request_to_api(session, payload, api_url)
df = pd.DataFrame(response["data"])
display(df)

,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,languages,page_id,page_name,publisher_platforms,target_ages,target_gender,target_locations,total_reach_by_location
0,1398240104596751,2025-08-17,[🌟 Say Goodbye to Neuropathy Pain with Our Neu...,[healthhubzz.com],[🌟 Rapid Pain Relief: Quickly soothe nerve pai...,[🌟 Instant Relief with Our Nerve Healing Cream...,2025-08-17,2025-08-18,"[{'country': 'BE', 'age_gender_breakdowns': [{...",[en],189764851743163,Kelly Shutt,[facebook],"[18, 65]",All,"[{'name': 'European Economic Area (EEA)', 'num...","[{'key': 'EU', 'value': 746}, {'key': 'GB', 'v..."
1,1679127252750216,2025-08-17,[🌟 Say Goodbye to Neuropathy Pain with Our Neu...,[healthhubzz.com],[🌟 Rapid Pain Relief: Quickly soothe nerve pai...,[🌟 Instant Relief with Our Nerve Healing Cream...,2025-08-17,2025-08-19,"[{'country': 'CY', 'age_gender_breakdowns': [{...",[en],189764851743163,Kelly Shutt,[facebook],"[18, 65]",All,"[{'name': 'European Economic Area (EEA)', 'num...","[{'key': 'EU', 'value': 892}, {'key': 'GB', 'v..."
2,1775545670017730,2025-08-17,[🌟 Say Goodbye to Neuropathy Pain with Our Neu...,[healthhubzz.com],[🌟 Rapid Pain Relief: Quickly soothe nerve pai...,[🌟 Instant Relief with Our Nerve Healing Cream...,2025-08-17,2025-08-18,"[{'country': 'IE', 'age_gender_breakdowns': [{...",[en],189764851743163,Kelly Shutt,[facebook],"[18, 65]",All,"[{'name': 'European Economic Area (EEA)', 'num...","[{'key': 'EU', 'value': 184}, {'key': 'GB', 'v..."
3,3699250353711993,2025-08-17,[🌟 Say Goodbye to Neuropathy Pain with Our Neu...,[healthhubzz.com],[🌟 Rapid Pain Relief: Quickly soothe nerve pai...,[🌟 Instant Relief with Our Nerve Healing Cream...,2025-08-17,2025-08-18,"[{'country': 'PL', 'age_gender_breakdowns': [{...",[en],189764851743163,Kelly Shutt,[facebook],"[18, 65]",All,"[{'name': 'European Economic Area (EEA)', 'num...","[{'key': 'EU', 'value': 121}, {'key': 'GB', 'v..."
4,23867551009608298,2025-08-17,[🌟 Say Goodbye to Neuropathy Pain with Our Neu...,[healthhubzz.com],[🌟 Rapid Pain Relief: Quickly soothe nerve pai...,[🌟 Instant Relief with Our Nerve Healing Cream...,2025-08-17,2025-08-18,"[{'country': 'DE', 'age_gender_breakdowns': [{...",[en],189764851743163,Kelly Shutt,[facebook],"[18, 65]",All,"[{'name': 'European Economic Area (EEA)', 'num...","[{'key': 'EU', 'value': 177}, {'key': 'GB', 'v..."
5,739620682176679,2025-08-17,[🌟 Say Goodbye to Neuropathy Pain with Our Neu...,[healthhubzz.com],[🌟 Rapid Pain Relief: Quickly soothe nerve pai...,[🌟 Instant Relief with Our Nerve Healing Cream...,2025-08-17,2025-08-27,"[{'country': 'AT', 'age_gender_breakdowns': [{...",[en],189764851743163,Kelly Shutt,[facebook],"[18, 65]",All,"[{'name': 'European Economic Area (EEA)', 'num...","[{'key': 'EU', 'value': 28069}, {'key': 'GB', ..."
6,777150081484411,2025-08-17,[🌟 Say Goodbye to Neuropathy Pain with Our Neu...,[healthhubzz.com],[🌟 Rapid Pain Relief: Quickly soothe nerve pai...,[🌟 Instant Relief with Our Nerve Healing Cream...,2025-08-17,2025-08-27,"[{'country': 'AT', 'age_gender_breakdowns': [{...",[en],189764851743163,Kelly Shutt,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'European Economic Area (EEA)', 'num...","[{'key': 'EU', 'value': 9985}, {'key': 'GB', '..."
7,1462225944816126,2025-08-17,[🌟 Say Goodbye to Neuropathy Pain with Our Neu...,[healthhubzz.com],[🌟 Rapid Pain Relief: Quickly soothe nerve pai...,[🌟 Instant Relief with Our Nerve Healing Cream...,2025-08-17,2025-08-27,"[{'country': 'AT', 'age_gender_breakdowns': [{...",[en],189764851743163,Kelly Shutt,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'European Economic Area (EEA)', 'num...","[{'key': 'EU', 'value': 14291}, {'key': 'GB', ..."
8,2269876786796552,2025-08-17,[🌟 Say Goodbye to Neuropathy Pain with Our Neu...,[healthhubzz.com],[🌟 Rapid Pain Relief: Quic

**OC15: Does the platform indicate whether ad creatives were generated using artificial intelligence?**

*This item verifies whether the platform flags ad campaigns in which AI played a role in producing the content. The assessment should review ad records to confirm the presence of a field or label indicating the use of AI in ad production.*
 


There's no field that indicates if an ad was created using AI. A complete list of available fields can be found in: https://developers.facebook.com/docs/graph-api/reference/archived-ad/

### CONSISTENCY

**OC23: Does the data retrieved by the API reflect what is displayed on the platform’s ad repository GUI?**

*This item verifies whether the data returned by the platform’s ad repository API corresponds to the information displayed on its GUI in all its levels of detail. It should be possible to identify in the API response information such as authorship, complete content, and other campaigning information (e.g., amount spent, impressions reached). The assessment should compare API responses with the GUI to confirm that key elements—such as authorship, full content, and campaign information (e.g., spending, impressions)—are consistently included.*


In [38]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
page_id = "213571821842759" # SET VALUE
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["NL"],  # USING A SINGLE COUNTRY TO MAKE IT EASIER TO VERIFY ON THE GUI
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_page_ids": page_id,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
print(response["data"][0])

{'id': '501914412268648', 'ad_creation_time': '2024-12-19', 'ad_creative_bodies': ['✨️️️️️️️ Stop overthinking and start living today ✨️️️️️️️\n\n🎯 Boost confidence: Develop strategies for better decision making and self-esteem\n💫 Eliminate negative thoughts: Learn how to identify and reduce toxic thought patterns\n⌛ Live in the present moment: Become calm, happy, and trust your intuition'], 'ad_creative_link_captions': ['mindway.app'], 'ad_creative_link_titles': ['Chronic Overthinking Test 👉'], 'ad_delivery_start_time': '2024-12-23', 'ad_delivery_stop_time': '2025-02-22', 'age_country_gender_reach_breakdown': [{'country': 'PT', 'age_gender_breakdowns': [{'age_range': '45-54', 'male': 263, 'female': 103, 'unknown': 5}, {'age_range': '65+', 'male': 33, 'female': 14}, {'age_range': '55-64', 'male': 83, 'female': 26, 'unknown': 1}, {'age_range': '18-24', 'male': 200, 'female': 52, 'unknown': 4}, {'age_range': '35-44', 'male': 278, 'female': 101, 'unknown': 5}, {'age_range': '25-34', 'male

**OC24: Are the results returned by the platform consistently reproducible?**


*This item verifies whether data accessed and extracted via the platform’s ad repository at a given time is consistent with other collections performed similarly, including cases where content was deleted in the interim. The assessment should perform repeated queries to confirm reproducibility of results.*

Test instructions: Please, develop a test that collects ads for 5 times using the same request parameters and the same endpoint. Save each response in separate files.

In [39]:
# Use this cell, or more, to develop a process that collects ads more than one time using the same request parameters.
# Describe the endpoint and the search parameters used.
# Please add each response for each time a request is made in a file and save it in the data folder of this repository.
# The files naming pattern should be as stated below:
total_runs = 5 
for idx in range(total_runs):
    sleep(10)
    index = idx + 1
    timestamp = datetime.now().strftime("%Y-%m-%dT%H-%M-%S-%f")
    filename = f"eu-meta-ads-question-24-run-{index}-{timestamp}"

    fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
    api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
    query = "health" # SET VALUE
    payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
    response = request_to_api(session, payload, api_url)

    outfile = os.path.join(FILEPATH, f"{filename}.json")

    with open(outfile, "w") as fout:
        json.dump(response, fout)

In [40]:
# Read files here to compare
pattern = os.path.join(FILEPATH, "eu-meta-ads-question-24*")
files = sorted(glob.glob(pattern))
all_files = []
for idx, file in enumerate(files):
    with open(file) as fin:
        data = json.load(fin)
    df = pd.DataFrame(data["data"])
    df["filename"] = os.path.basename(file)
    all_files.append(df)
df_complete = pd.concat(all_files, ignore_index=True)
df_agg = df_complete.groupby("id").agg(total_occurance=("id", "count"),first_occurance_file=("filename", "min"), last_occurance_file=("filename", "max"),start_delivery_date_max=("ad_delivery_start_time", "max"))
display(df_agg)
    

,total_occurance,first_occurance_file,last_occurance_file,start_delivery_date_max
id,,,,
1028188812658253,5,eu-meta-ads-question-24-run-1-2025-11-21T13-41...,eu-meta-ads-question-24-run-5-2025-11-21T13-42...,2024-12-21
1089953079089302,5,eu-meta-ads-question-24-run-1-2025-11-21T13-41...,eu-meta-ads-question-24-run-5-2025-11-21T13-42...,2024-12-21
1091260115646259,5,eu-meta-ads-question-24-run-1-2025-11-21T13-41...,eu-meta-ads-question-24-run-5-2025-11-21T13-42...,2024-12-25
1092775425488102,5,eu-meta-ads-question-24-run-1-2025-11-21T13-41...,eu-meta-ads-question-24-run-5-2025-11-21T13-42...,2024-12-21
1093022218791684,5,eu-meta-ads-question-24-run-1-2025-11-21T13-41...,eu-meta-ads-question-24-run-5-2025-11-21T13-42...,2024-12-21
1149767699841006,5,eu-meta-ads-question-24-run-1-2025-11-21T13-41...,eu-meta-ads-question-24-run-5-2025-11-21T13-42...,2024-12-23
1198154008406828,5,eu-meta-ads-question-24-run-1-2025-11-21T13-41...,eu-meta-ads-question-24-run-5-2025-11-21T13-42...,2024-12-19
1238312724100050,5,eu-meta-ads-question-24-run-1-2025-11-21T13-41...,eu-meta-ads-question-24-run-5-2025-11-21T13-42...,2024-12-23
1266941774632571,5,eu-meta-ads-question-24-run-1-2025-11-21T13-41...,eu-meta-ads-question-24-run-5-2025-11-21T13-42...,2024-12-23


**OC25: Is the data returned by the platform consistent with the parameters and filters used in the request?**

*This item verifies whether the data retrieved through the ad repository accurately reflects the parameters and filters specified at the time of the request. The assessment should run test queries with different filters to confirm that results consistently match the requested conditions.*

Test instructions: Please, develop a process that request data using different parameter to the same endpoint. If possible, test at least 5 different parameters/filters. If the platform provides less than 5, use all available parameters/filters.

Save each response in separate files. 

In [46]:
# Use this cell, or more, to develop a process that collects ads more than one time using different parameters.
# Describe the endpoint and the search parameters used.
# Please add each response for each time a request is made in a file and save it in the data folder of this repository.
# The files naming pattern should be as stated below:

#FEEL FREE TO CHANGE PARAMETERS
parameters = [{"ad_delivery_date_min": "2025-11-01"}, 
                {"bylines": ["Walking Yoga"]}, 
                {"delivery_by_region": ["Dublin"]}, 
                {"estimated_audience_size_min":1000}, 
                {"languages": ["en"]}]
for idx, param in enumerate(parameters):
    sleep(1)
    index = idx + 1 
    timestamp = datetime.now().strftime("%Y-%m-%dT%H-%M-%S-%f")
    filename = f"eu-meta-ads-question-25-run-{index}-{list(param.keys())[0]}-{timestamp}"
    
    fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
    api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
    query = "health" # Feel free to change value
    payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
    payload.update(param)
    response = request_to_api(session, payload, api_url)

    with open(f"{FILEPATH}/{filename}.json", "w") as fout:
        json.dump(response, fout)

In [47]:
# ad_delivery_date_min > 2025-11-01
filename = "eu-meta-ads-question-25-run-1-ad_delivery_date_min-2025-11-21T13-48-07-873203.json" # SET VALUE
with open(f"{FILEPATH}/{filename}") as fin:
    data = json.load(fin)
df = pd.DataFrame(data["data"])
df["ad_delivery_start_time"] = pd.to_datetime(df["ad_delivery_start_time"])
df["ad_delivery_stop_time"] = pd.to_datetime(df["ad_delivery_stop_time"])
schema = pa.DataFrameSchema(
    {
        "ad_delivery_stop_time": pa.Column(
            pa.DateTime, 
            checks=pa.Check.greater_than_or_equal_to("2025-11-01"),
            nullable=False,  
            title="Ad delivery start time"
        )
    },
    strict=False
)
try:
    schema.validate(df, lazy=True)
    print("✅ Validation Successful: All ads ran after the set ad_delivery_date_min parameter.")
except SchemaErrors as err:
    print("❌ Validation Failed: Found ads that ran earlier than the set ad_delivery_date_min parameter")
    print(err)




✅ Validation Successful: All ads ran after the set ad_delivery_date_min parameter.


In [48]:
# bylines: Walking Yoga
filename = "eu-meta-ads-question-25-run-2-bylines-2025-11-21T13-48-10-539689.json" # SET VALUE
with open(f"{FILEPATH}/{filename}") as fin:
    data = json.load(fin)
if "data" in data:
    df = pd.DataFrame(data["data"])

    schema = pa.DataFrameSchema(
        {
            "bylines": pa.Column(
                checks=pa.Check.equal_to("Labour Future"),
                nullable=False,  
                title="Bylines"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have funders information as set in bylines parameter.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Found ads that are not from funders as set in bylines parameter")
        print(err)




In [49]:
# delivery_by_region: Dublin
def extract_regions(x):
    if isinstance(x, list) and len(x) > 0: 
        return [d['region'] for d in x]
    return x

def check_delivery_region(regions):
    return regions.apply(lambda region_list: "Dublin" in region_list)

filename = "eu-meta-ads-question-25-run-3-delivery_by_region-2025-11-21T13-48-11-707899.json" # SET VALUE
with open(f"{FILEPATH}/{filename}") as fin:
    data = json.load(fin)

if "data" in data:
    df = pd.DataFrame(data["data"])
    df["ad_delivery_regions"] = df["delivery_by_region"].apply(extract_regions)

    schema = pa.DataFrameSchema(
        {
            "ad_delivery_regions": pa.Column(
                checks=pa.Check(check_delivery_region),
                nullable=False,  
                title="Ad delivery regions"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have region information as set in delivery_by_region parameter.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Found ads that don't have or are not from the regions as set in delivery_by_region parameter")
        print(err)





In [50]:
# estimated_audience_size_min > 500
filename = "eu-meta-ads-question-25-run-4-estimated_audience_size_min-2025-11-21T13-48-12-892369.json" # SET VALUE
with open(f"{FILEPATH}/{filename}") as fin:
    data = json.load(fin)

if "data" in data:
    df = pd.json_normalize(data["data"])
    df["estimated_audience_size.lower_bound"] = df["estimated_audience_size.lower_bound"].astype("int64") # NEEDS TO DECLARE INT64, AS SOME OF THEM CAME AS INT32
   
    schema = pa.DataFrameSchema(
        {
            "estimated_audience_size.lower_bound": pa.Column(
                pa.Int, 
                checks=pa.Check.greater_than_or_equal_to(500),
                nullable=False,  
                title="Estimated audience size - Lower"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have audience size greater than the set estimated_audience_size_min parameter.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Found ads that have audience size lower than the set estimated_audience_size_min parameter")
        print(err)

In [51]:
# language: en
filename = "eu-meta-ads-question-25-run-5-languages-2025-11-21T13-48-14-064138.json"
with open(f"{FILEPATH}/{filename}") as fin:
    data = json.load(fin)

if "data" in data:
    df = pd.DataFrame(data["data"])
    df["languages"] = df["languages"].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x)


    schema = pa.DataFrameSchema(
        {
            "languages": pa.Column(
                checks=pa.Check.equal_to("en"),
                nullable=False,  
                title="Languages"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have information regarding language and it's the correct language set in language parameter.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Found ads that don't have or are not from the language set in the language parameter")
        print(err)





✅ Validation Successful: All ads have information regarding language and it's the correct language set in language parameter.


### RELEVANCE

**OC26: Does the platform allow the use of temporal filters to retrieve data on ads?**

*This item verifies whether the ad repository allows filtering ad campaign data based on the time period during which the ads were served. The assessment should test queries with temporal filters to confirm that results accurately reflect the specified date ranges.*



In [52]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,ad_creation_time,ad_delivery_start_time,ad_delivery_stop_time"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True,
            "ad_delivery_date_max": "2025-10-15", # FEEL FREE TO CHANGE DATES
            "ad_delivery_date_min": "2025-10-01" # FEEL FREE TO CHANGE DATES
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No ads were found")
    print(response)


Got 25 ads in a single request


,id,ad_creation_time,ad_delivery_start_time,ad_delivery_stop_time
0,1386801456407477,2025-10-13,2025-10-13,2025-10-14
1,1477118690204649,2025-10-13,2025-10-13,2025-10-16
2,781046464730300,2025-10-13,2025-10-14,2025-10-18
3,1391798712277538,2025-10-13,2025-10-14,2025-10-18
4,1725085064820431,2025-10-13,2025-10-14,2025-10-18
5,1772347750141394,2025-10-13,2025-10-14,2025-10-18
6,1134956794875032,2025-10-04,2025-10-10,2025-10-11
7,1478511413383089,2025-10-04,2025-10-05,2025-10-07
8,2344572639294524,2025-10-04,2025-10-05,2025-10-07
9,3337165913105859,2025-10-04,2025-10-10,2025-10-11


In [53]:
df = pd.DataFrame(response["data"])
df["ad_delivery_start_time"] = pd.to_datetime(df["ad_delivery_start_time"])
start_date = datetime(year=2025, month=10, day=1)
end_date = datetime(year=2025, month=10, day=15)

schema = pa.DataFrameSchema({
    "ad_delivery_start_time": pa.Column(
        pa.DateTime,
        pa.Check(lambda s: s.between(start_date, end_date, inclusive="both"), 
                 name="DateRangeCheck"),
        nullable=False
    )
})
try:
    schema.validate(df, lazy=True)
    print("✅ Validation Successful: All ads are between date range.")

except SchemaErrors as err:
    print("❌ Validation Failed: Found ads outside date range.")
    print(err) 

✅ Validation Successful: All ads are between date range.


**OC27: Does the platform allow filtering advertising data by ad category?**

*This item verifies whether the ad repository allows filtering ad campaign data based on any possible categories assigned at the time of ad creation. The assessment should run test queries with category filters to confirm that results align with the selected classifications.*


In [54]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "EMPLOYMENT_ADS", # FEEL FREE TO CHANGE CATEGORY
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in EMPLOYMENT_ADS in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found in EMPLOYMENT_ADS")
    print(response)

Got 25 in EMPLOYMENT_ADS in a single request


,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,languages,page_id,page_name,publisher_platforms,target_ages,target_gender,target_locations,total_reach_by_location,ad_creative_link_descriptions
0,823490390655448,2025-11-20,[Discover this opportunity! 🔥],[bestagencywork.com],[Discover this opportunity! 🔥],2025-11-21,2025-11-21,"[{'country': 'DE', 'age_gender_breakdowns': [{...",[en],615127158344887,Curlia Haeper,"[facebook, instagram]","[18, 65]",All,"[{'name': 'Guadeloupe', 'num_obfuscated': 0, '...","[{'key': 'EU', 'value': 54}, {'key': 'GB', 'va...",NaN
1,838074162485111,2025-11-20,[Pest control technicians can earn industry ce...,[hometalk.com],[Pest Control Career Paths And Requirements Gu...,2025-11-20,2025-11-21,"[{'country': 'SE', 'age_gender_breakdowns': [{...",[en],774075372456850,Looping - Services,"[facebook, instagram, messenger]","[18, 65]",All,"[{'name': 'Guadeloupe', 'num_obfuscated': 0, '...","[{'key': 'EU', 'value': 38}, {'key': 'GB', 'va...",[The pest control industry plays a crucial rol...
2,1182720533789199,2025-11-20,[Descubre más información sobre Trabajar en Es...,[gethappyday.com],[Leer más sobre Trabajar En España Como Peluqu...,2025-11-20,NaN,"[{'country': 'ES', 'age_gender_breakdowns': [{...",[es],854290684431002,Healing Roots Health,"[facebook, instagram, messenger]","[18, 65]",All,"[{'name': 'Guadeloupe', 'num_obfuscated': 0, '...","[{'key': 'EU', 'value': 1079}, {'key': 'GB', '...",[Saber más]
3,843096401415057,2025-11-20,[Descubre más información sobre Trabajar en Es...,[gethappyday.com],[Leer más sobre Trabajar En España Como Peluqu...,2025-11-20,NaN,"[{'country': 'FR', 'age_gender_breakdowns': [{...",[es],854290684431002,Healing Roots Health,"[facebook, instagram, messenger]","[18, 65]",All,"[{'name': 'Guadeloupe', 'num_obfuscated': 0, '...","[{'key': 'EU', 'value': 1597}, {'key': 'GB', '...",[Saber más]
4,1157749152691621,2025-11-20,[Interessiert an einem Teilzeitjob?🕑 Lesen Sie...,[gethappyday.com],[Lesen Sie mehr über: Teilzeit Pflegehelfer/in...,2025-11-20,2025-11-21,"[{'country': 'DE', 'age_gender_breakdowns': [{...",[de],854290684431002,Healing Roots Health,"[facebook, instagram, messenger]","[18, 65]",All,"[{'name': 'Guadeloupe', 'num_obfuscated': 0, '...","[{'key': 'EU', 'value': 68}]",[Learn more]
5,856048903574084,2025-11-20,[Interessiert an einem Teilzeitjob?🕑 Lesen Sie...,[gethappyday.com],[Lesen Sie mehr über: Teilzeit Pflegehelfer/in...,2025-11-20,2025-11-21,"[{'country': 'SK', 'age_gender_breakdowns': [{...",[de],854290684431002,Healing Roots Health,"[facebook, instagram, messenger]","[18, 65]",All,"[{'name': 'Guadeloupe', 'num_obfuscated': 0, '...","[{'key': 'EU', 'value': 1604}, {'key': 'GB', '...",[Learn more]
6,1155133829937289,2025-11-20,[Learn More About Work In Dubai as a Fitness-c...,[healthandwellnesnews.com],[Fitness Coaching in Dubai — What to Know],2025-11-20,NaN,"[{'country': 'FR', 'age_gender_breakdowns': [{...",NaN,102214368694427,Ciencia Espacial,"[facebook, instagram, audience_network, messen...","[18, 65]",All,"[{'name': 'Italy', 'num_obfuscated': 0, 'type'...","[{'key': 'EU', 'value': 4}, {'key': 'GB', 'val...",[Health and Wellness News]
7,830525349593401,2025-11-19,[Explore the guide about informations on work ...,[morehackz.com],[Explore U.S. Work Program Options],2025-11-19,NaN,"[{'country': 'SK', 'age_gender_breakdowns': [{...",[en],587519667778680,Rounder Employment Route,"[facebook, instagram]","[18, 65]",All,"[{'name': 'Guadeloupe', 'num_obfuscated': 0, '...","[{'key': 'EU', 'value': 16}, {'key': 'GB', 'va...",NaN
8,1529591385134276,2025-11-19,[Looking for a New Opportunity? Discover more ...,[gethappyday.com],[Learn About Environmental Research Assistant ...,2025-11-20,NaN,"[{'country': 'CZ', 'age_gender_breakdowns': [{...",[en],854290684431002,Healing Roots Health,"[facebook, instagram, messenger]","[18, 65]",All,"[{'name': 'Guadelou

**OC28: Does the platform allow filtering advertising data by geographic location?**

*This item assesses whether the ad repository allows filtering data by one or more subnational geographic locations where the ads were served. The assessment should test queries with location filters to confirm that results match the specified areas.*

In [55]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,delivery_by_region"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "delivery_by_region": ["Dublin"], # FEEL FREE TO CHANGE REGION
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found")
    print(response)

No data found
{'error': {'message': 'An unknown error has occurred.', 'type': 'OAuthException', 'code': 1, 'fbtrace_id': 'AFsMmZWVJDszVZYU7eseh7j'}}


In [56]:
# delivery_by_region: Dublin
def extract_regions(x):
    if isinstance(x, list) and len(x) > 0:
        return [d['region'] for d in x]
    return x

def check_delivery_region(regions):
    return regions.apply(lambda region_list: "Dublin" in region_list)

if "data" in response:
    df = pd.DataFrame(response["data"])
    df["ad_delivery_regions"] = df["delivery_by_region"].apply(extract_regions)

    schema = pa.DataFrameSchema(
        {
            "ad_delivery_regions": pa.Column(
                checks=pa.Check(check_delivery_region),
                nullable=False,  
                title="Ad delivery regions"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads came from the region set in the geographic location filter.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Found ads that don't have or are not from the region set in the geographic location filter.")
        print(err)





### ACCURACY

**OC29: Does the platform provide age and gender data on the audiences of ads?**

*This item verifies whether the platform provides data on the age and gender of audiences reached. The assessment should review the ad records to confirm that these breakdowns are available and consistently reported*

In [57]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,age_country_gender_reach_breakdown,demographic_distribution,target_ages,target_gender"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found")
    print(response)

Got 25 in a single request


,id,age_country_gender_reach_breakdown,target_ages,target_gender
0,468492669268944,"[{'country': 'PT', 'age_gender_breakdowns': [{...","[18, 65]",All
1,540462482317039,"[{'country': 'SE', 'age_gender_breakdowns': [{...","[18, 65]",All
2,927457872686945,"[{'country': 'ES', 'age_gender_breakdowns': [{...","[18, 65]",All
3,1149767699841006,"[{'country': 'EE', 'age_gender_breakdowns': [{...","[18, 65]",All
4,1198154008406828,"[{'country': 'EE', 'age_gender_breakdowns': [{...","[18, 65]",All
5,1645469656179516,"[{'country': 'IT', 'age_gender_breakdowns': [{...","[18, 65]",All
6,1681368622408808,"[{'country': 'FR', 'age_gender_breakdowns': [{...","[18, 65]",All
7,1699924333908780,"[{'country': 'RO', 'age_gender_breakdowns': [{...","[18, 65]",All
8,1990654091358510,"[{'country': 'CY', 'age_gender_breakdowns': [{...","[18, 65]",All
9,573867262268518,"[{'country': 'MQ', 'age_gender_breakdowns': [{...","[18, 65]",All


In [58]:
if "data" in response:
    df = pd.json_normalize(response["data"])
    schema = pa.DataFrameSchema(
        {
            "age_country_gender_reach_breakdown": pa.Column( 
                nullable=False,  
                title="Age Country Gender Reach Breakdown"
            ),
            "target_ages": pa.Column(
                nullable=False,  
                title="Target Ages"
            ),
            "target_gender": pa.Column(
                nullable=False,  
                title="Target Ages"
            ),
            "demographic_distribution": pa.Column(
                nullable=False,  
                title="Target Ages"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have information about age and gender about ad audience.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Missing ads information about age and gender.")
        print(err)


❌ Validation Failed: Missing ads information about age and gender.
{
    "SCHEMA": {
        "COLUMN_NOT_IN_DATAFRAME": [
            {
                "schema": null,
                "column": null,
                "check": "column_in_dataframe",
                "error": "column 'demographic_distribution' not in dataframe. Columns in dataframe: ['id', 'age_country_gender_reach_breakdown', 'target_ages', 'target_gender']"
            }
        ]
    }
}


**OC30: Does the platform provide subnational geographic data on the audience reached by ads?**

*This item verifies whether the platform provides data on the subnational geographic location of audiences reached. The assessment should review the ad records to confirm that such location breakdowns are available and consistently reported.*

In [59]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,delivery_by_region,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}" 
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found")
    print(response)

Got 25 in a single request


,id,target_locations,total_reach_by_location
0,468492669268944,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 192}]"
1,540462482317039,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 4}]"
2,927457872686945,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 10002}]"
3,1149767699841006,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 2229}]"
4,1198154008406828,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 5213}]"
5,1645469656179516,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 26032}]"
6,1681368622408808,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 8067}]"
7,1699924333908780,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 74}]"
8,1990654091358510,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ...","[{'key': 'EU', 'value': 6337}]"
9,573867262268518,"[{'name': 'Guadeloupe', 'num_obfuscated': 0, '...","[{'key': 'EU', 'value': 18506}]"


In [60]:
def check_region_granularity(target_locations): 
    subnational_granularity = False
    if isinstance(target_locations, list) and len(target_locations) > 0:
        for location in target_locations:
            locations = location.get("name").split(",")
            if len(locations) > 1:
                subnational_granularity = True
                break
    return subnational_granularity

def validate_granularity_column(s):
    return s.apply(check_region_granularity)

if "data" in response:
    df = pd.json_normalize(response["data"])
    schema = pa.DataFrameSchema(
        {
            "delivery_by_region": pa.Column( 
                nullable=False,  
                title="Delivery by region"
            ),
            "target_locations": pa.Column(
                pa.Object,
                pa.Check(validate_granularity_column, error="Ad lacks required subnational granularity."),
                nullable=False,  
                title="Target location"
            ),
            "total_reach_by_location": pa.Column(
                nullable=False,  
                title="Total reach by location"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have subnational geographic information about ad audience.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Missing ads subnational geographic information.")
        print(err)


❌ Validation Failed: Missing ads subnational geographic information.
{
    "SCHEMA": {
        "COLUMN_NOT_IN_DATAFRAME": [
            {
                "schema": null,
                "column": null,
                "check": "column_in_dataframe",
                "error": "column 'delivery_by_region' not in dataframe. Columns in dataframe: ['id', 'target_locations', 'total_reach_by_location']"
            }
        ]
    },
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": null,
                "column": "target_locations",
                "check": "Ad lacks required subnational granularity.",
                "error": "Column 'target_locations' failed element-wise validator number 0: Ad lacks required subnational granularity. failure cases: [{'name': 'Ireland', 'num_obfuscated': 0, 'type': 'countries', 'excluded': False}, {'name': 'Italy', 'num_obfuscated': 0, 'type': 'countries', 'excluded': False}, {'name': 'French Guiana', 'num_obfuscated': 0, 'typ

**OC31: Does the platform include data on audience targeting criteria defined by advertisers?**

*This item verifies whether the platform provides data on audience targeting criteria defined by the advertiser when publishing ads (e.g., demographic and geographic segments, as well as interests, attitudes, behaviors, and keywords). The assessment should review ad records to confirm that these targeting parameters are available and consistently reported.*


There's no field that indicates the targeting defined by the advertiser. A complete list of available fields can be found in: https://developers.facebook.com/docs/graph-api/reference/archived-ad/

In [61]:
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,estimated_audience_size,target_ages,target_gender,target_locations"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found")
    print(response)

Got 25 in a single request


,id,target_ages,target_gender,target_locations
0,468492669268944,"[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ..."
1,540462482317039,"[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ..."
2,927457872686945,"[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ..."
3,1149767699841006,"[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ..."
4,1198154008406828,"[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ..."
5,1645469656179516,"[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ..."
6,1681368622408808,"[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ..."
7,1699924333908780,"[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ..."
8,1990654091358510,"[18, 65]",All,"[{'name': 'Ireland', 'num_obfuscated': 0, 'typ..."
9,573867262268518,"[18, 65]",All,"[{'name': 'Guadeloupe', 'num_obfuscated': 0, '..."


In [62]:
if "data" in response:
    df = pd.json_normalize(response["data"])
    schema = pa.DataFrameSchema(
        {
            "target_ages": pa.Column(
                nullable=False,
                title="Target Ages"
            ),
            "target_gender": pa.Column(
                nullable=False,
                title="Target Gender"
            ),
            "target_locations": pa.Column(
                nullable=False,
                title="Target Locations"
            ),
        },
        strict=False
    )

    try:
        schema.validate(df, lazy=True)
        print("✅ OC31: Validation successful – for this sample, all ads include age, gender, and geographic targeting fields defined by the advertiser.")
    except SchemaErrors as err:
        print("❌ OC31: Validation failed – some ads are missing age, gender, or geographic targeting information.")
        print(err)
else:
    print("No data found")
    print(response)


✅ OC31: Validation successful – for this sample, all ads include age, gender, and geographic targeting fields defined by the advertiser.


**OC32: Does the platform provide granular volume ranges for ad impressions?**

*This item verifies whether the ad repository provides impression values for ads, using ranges that closely approximate the actual numbers. Intervals should be no larger than 10% of the upper bound of the value range they represent. For example, values up to 1,000 impressions should be displayed in intervals no larger than 100; between 1,000 and 10,000 in intervals no larger than 1,000; between 10,000 and 100,000 in intervals no larger than 10,000; between 100,000 and 1 million or above, in intervals no larger than 100,000. The assessment should measure whether the reported intervals remain within this threshold across the different value ranges using the platform’s documentation or available data interfaces.*


In [63]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,impressions"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": False
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data in response")
    print(response)


Got 25 ads in a single request


,id
0,468492669268944
1,540462482317039
2,927457872686945
3,1149767699841006
4,1198154008406828
5,1645469656179516
6,1681368622408808
7,1699924333908780
8,1990654091358510
9,573867262268518


In [65]:
if response["data"]:
    df = pd.json_normalize(response["data"])
    required_cols = ["impressions.upper_bound", "impressions.lower_bound"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required impressions columns: {missing}")
    df["impressions.upper_bound"]= df["impressions.upper_bound"].astype(int)
    df["percentage"] = round((df["impressions.upper_bound"].astype(int) - df["impressions.lower_bound"].astype(int) ) / df["impressions.upper_bound"].astype(int)  * 100, 1)
    schema = pa.DataFrameSchema(
        {
            "percentage": pa.Column( 
                pa.Float,
                pa.Check.less_than_or_equal_to(10),
                nullable=False, 
                title="Interval between impressions values percentage"
            ),
            "impressions.upper_bound": pa.Column(
                nullable=False,  
                title="Impressions - Upper"
            ),
            "impressions.lower_bound": pa.Column(
                nullable=False,  
                title="Impressions - Lower"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have impressions granularity in the expected range.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Impressions granularity between lower and upper bound are larger than 10% of the upper bound")
        print(err)


ValueError: Missing required impressions columns: ['impressions.upper_bound', 'impressions.lower_bound']

**OC33: Does the platform provide granular investment ranges for ad spending?**

*This item verifies whether the ad repository provides spending values for ads, using ranges that closely approximate the actual amounts. Intervals should be no larger than 10% of the upper bound of the value range they represent. For example, values up to $100 should be displayed in intervals no larger than $10; between $100 and $1,000 in intervals no larger than $100; and between $1,000 and $10,000 in intervals no larger than $1,000. The assessment should measure whether the reported intervals remain within this threshold across the different value ranges using the platform’s documentation or available data interfaces.*


In [66]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,spend"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "health" # SET VALUE
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": eu_countries_iso_list,
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": False
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response['data'])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data in response")
    print(response)


Got 25 ads in a single request


,id
0,468492669268944
1,540462482317039
2,927457872686945
3,1149767699841006
4,1198154008406828
5,1645469656179516
6,1681368622408808
7,1699924333908780
8,1990654091358510
9,573867262268518


In [67]:
if response["data"]:
    df = pd.json_normalize(response["data"])
    required_cols = ["spend.upper_bound", "spend.lower_bound"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required spend columns: {missing}")
    df["spend.upper_bound"]= df["spend.upper_bound"].astype(int)
    df["spend.lower_bound"]= df["spend.lower_bound"].astype(int)
    df["percentage"] = round((df["spend.upper_bound"] - df["spend.lower_bound"] ) / df["spend.upper_bound"]  * 100, 1)
    schema = pa.DataFrameSchema(
        {
            "percentage": pa.Column( 
                pa.Float,
                pa.Check.less_than_or_equal_to(10),
                nullable=False, 
                title="Interval between spend values percentage"
            ),
            "spend.upper_bound": pa.Column(
                nullable=False,  
                title="Spend - Upper"
            ),
            "spend.lower_bound": pa.Column(
                nullable=False,  
                title="Spend - Lower"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have spend granularity in the expected range.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Spend granularity between lower and upper bound are larger than 10% of the upper bound")
        print(err)


ValueError: Missing required spend columns: ['spend.upper_bound', 'spend.lower_bound']